# Demo NER inference

Notebook tối giản để inference một danh sách câu đầu vào bằng 3 chế độ:

- `gazetteer`: match bằng `data/preprocessed/expanded_gazetteer`.
- `model`: predict bằng GLiNER fine-tuned ở `data/models/gliner_traffic_ner/final_model`.
- `hybrid`: hợp nhất kết quả của gazetteer và GLiNER.

Mặc định notebook chạy `gazetteer` để demo mở lên là chạy được ngay. Nếu muốn dùng `model` hoặc `hybrid`, env đang chạy notebook cần có package `gliner`.

In [5]:
from pathlib import Path
import importlib.util
import sys
from pprint import pprint

# Chạy được cả khi mở notebook từ repo root hoặc từ thư mục ner_finetuning.
cwd = Path.cwd().resolve()
if (cwd / "data" / "preprocessed" / "expanded_gazetteer").exists():
    NER_DIR = cwd
elif (cwd / "ner_finetuning" / "data" / "preprocessed" / "expanded_gazetteer").exists():
    NER_DIR = cwd / "ner_finetuning"
else:
    raise FileNotFoundError("Không tìm thấy ner_finetuning/data/preprocessed/expanded_gazetteer")

GAZETTEER_ROOT = NER_DIR / "data" / "preprocessed" / "expanded_gazetteer"
MODEL_DIR = NER_DIR / "data" / "models" / "gliner_traffic_ner" / "final_model"
HAS_GLINER = importlib.util.find_spec("gliner") is not None

sys.path.insert(0, str(NER_DIR / "gazetteer_building"))

print("NER_DIR:", NER_DIR)
print("Gazetteer:", GAZETTEER_ROOT)
print("Model:", MODEL_DIR)
print("GLiNER installed:", HAS_GLINER)

NER_DIR: D:\Users\ADMIN\LocalData\Classworks\KLTN\ner_finetuning
Gazetteer: D:\Users\ADMIN\LocalData\Classworks\KLTN\ner_finetuning\data\preprocessed\expanded_gazetteer
Model: D:\Users\ADMIN\LocalData\Classworks\KLTN\ner_finetuning\data\models\gliner_traffic_ner\final_model
GLiNER installed: True


In [6]:
from gazetteer_building_core.matcher import load_aliases, find_matches

LABELS = [
    "ACTOR",
    "BEHAVIOR",
    "CONDITION",
    "DOCUMENT",
    "INFRASTRUCTURE",
    "VEHICLE",
    "VEHICLE_CONDITION_OR_EQUIPMENT",
]

ALIASES = load_aliases(GAZETTEER_ROOT)
_gliner_model = None


def predict_with_gazetteer(texts):
    """Match entity bằng expanded gazetteer."""
    results = []
    for i, text in enumerate(texts):
        entities = find_matches(text, ALIASES)
        results.append({"id": i, "text": text, "entities": entities})
    return results


def load_gliner_model(device="cpu"):
    """Lazy-load GLiNER để notebook vẫn chạy gazetteer-only nhanh."""
    global _gliner_model
    if _gliner_model is not None:
        return _gliner_model

    if importlib.util.find_spec("gliner") is None:
        raise RuntimeError(
            "Env hiện tại chưa có package 'gliner'. "
            "Cài vào env kltn bằng: conda run -n kltn pip install gliner"
        )
    if not MODEL_DIR.exists():
        raise FileNotFoundError(f"Không tìm thấy model: {MODEL_DIR}")

    from gliner import GLiNER

    _gliner_model = GLiNER.from_pretrained(str(MODEL_DIR)).to(device)
    return _gliner_model


def predict_with_model(texts, threshold=0.70, device="cpu"):
    """Predict entity bằng GLiNER fine-tuned."""
    model = load_gliner_model(device=device)
    try:
        pred_batches = model.batch_predict_entities(texts, LABELS, threshold=threshold)
    except Exception:
        pred_batches = [model.predict_entities(text, LABELS, threshold=threshold) for text in texts]

    results = []
    for i, (text, preds) in enumerate(zip(texts, pred_batches)):
        entities = []
        for pred in preds:
            entities.append({
                "text": pred.get("text"),
                "label": pred.get("label"),
                "start": pred.get("start"),
                "end": pred.get("end"),
                "confidence": float(pred.get("score", 0.0)),
                "source": "gliner",
            })
        results.append({"id": i, "text": text, "entities": entities})
    return results


def merge_entities(gazetteer_entities, model_entities):
    """Hợp nhất đơn giản: bỏ trùng chính xác theo span + label + text."""
    merged = []
    seen = set()
    for entity in [*gazetteer_entities, *model_entities]:
        key = (
            entity.get("start"),
            entity.get("end"),
            entity.get("label"),
            (entity.get("text") or entity.get("surface") or "").lower(),
        )
        if key in seen:
            continue
        seen.add(key)
        merged.append(entity)
    return sorted(merged, key=lambda e: (e.get("start") is None, e.get("start") or 0, e.get("end") or 0))


def predict_hybrid(texts, threshold=0.70, device="cpu"):
    """Chạy cả gazetteer và GLiNER rồi hợp nhất kết quả."""
    gazetteer_results = predict_with_gazetteer(texts)
    model_results = predict_with_model(texts, threshold=threshold, device=device)

    results = []
    for gaz, mod in zip(gazetteer_results, model_results):
        results.append({
            "id": gaz["id"],
            "text": gaz["text"],
            "entities": merge_entities(gaz["entities"], mod["entities"]),
            "gazetteer_entities": gaz["entities"],
            "model_entities": mod["entities"],
        })
    return results


def infer(texts, mode="gazetteer", threshold=0.70, device="cpu"):
    """mode: 'gazetteer', 'model', hoặc 'hybrid'."""
    if mode == "gazetteer":
        return predict_with_gazetteer(texts)
    if mode == "model":
        return predict_with_model(texts, threshold=threshold, device=device)
    if mode == "hybrid":
        return predict_hybrid(texts, threshold=threshold, device=device)
    raise ValueError("mode phải là 'gazetteer', 'model' hoặc 'hybrid'")


print(f"Loaded {len(ALIASES):,} gazetteer aliases")

Loaded 1,830 gazetteer aliases


In [7]:
texts = [
    "Người điều khiển xe mô tô không đội mũ bảo hiểm khi tham gia giao thông.",
    "Giấy phép lái xe phải còn thời hạn sử dụng và phù hợp với loại xe được điều khiển.",
    "Xe ô tô không chấp hành hiệu lệnh của đèn tín hiệu giao thông.",
]

# Chọn một trong ba mode: "gazetteer", "model", "hybrid".
# Notebook mặc định dùng gazetteer để chạy được ngay cả khi chưa cài gliner.
mode = "gazetteer"
threshold = 0.70
device = "cpu"

results = infer(texts, mode=mode, threshold=threshold, device=device)
pprint(results, width=140)

[{'entities': [{'canonical': 'xe mô tô',
                'confidence': 1.0,
                'end': 25,
                'entity_id': 'ent_0a4f8c808c24e49c',
                'label': 'VEHICLE',
                'source': 'gazetteer',
                'start': 17,
                'surface': 'xe mô tô'},
               {'canonical': 'không đội mũ bảo hiểm',
                'confidence': 1.0,
                'end': 47,
                'entity_id': 'ent_c0e53c13e27bd149',
                'label': 'BEHAVIOR',
                'source': 'gazetteer',
                'start': 26,
                'surface': 'không đội mũ bảo hiểm'}],
  'id': 0,
  'text': 'Người điều khiển xe mô tô không đội mũ bảo hiểm khi tham gia giao thông.'},
 {'entities': [{'canonical': 'giấy phép lái xe',
                'confidence': 1.0,
                'end': 16,
                'entity_id': 'ent_5251cf1b81c1763b',
                'label': 'DOCUMENT',
                'source': 'gazetteer',
                'start': 0,
      

In [8]:
# So sánh nhanh các chế độ có thể chạy trong env hiện tại.
modes = ["gazetteer", "model", "hybrid"] if HAS_GLINER else ["gazetteer"]

for current_mode in modes:
    print("\n" + "=" * 80)
    print("MODE:", current_mode)
    pprint(infer(texts, mode=current_mode, threshold=threshold, device=device), width=140)

if not HAS_GLINER:
    print("\nEnv hiện tại chưa có gliner nên cell này chỉ chạy mode='gazetteer'.")


MODE: gazetteer
[{'entities': [{'canonical': 'xe mô tô',
                'confidence': 1.0,
                'end': 25,
                'entity_id': 'ent_0a4f8c808c24e49c',
                'label': 'VEHICLE',
                'source': 'gazetteer',
                'start': 17,
                'surface': 'xe mô tô'},
               {'canonical': 'không đội mũ bảo hiểm',
                'confidence': 1.0,
                'end': 47,
                'entity_id': 'ent_c0e53c13e27bd149',
                'label': 'BEHAVIOR',
                'source': 'gazetteer',
                'start': 26,
                'surface': 'không đội mũ bảo hiểm'}],
  'id': 0,
  'text': 'Người điều khiển xe mô tô không đội mũ bảo hiểm khi tham gia giao thông.'},
 {'entities': [{'canonical': 'giấy phép lái xe',
                'confidence': 1.0,
                'end': 16,
                'entity_id': 'ent_5251cf1b81c1763b',
                'label': 'DOCUMENT',
                'source': 'gazetteer',
                '

D:\Users\ADMIN\miniconda3\envs\kltn\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_2436\174582180.py:50: FutureWarning: GLiNER.batch_predict_entities is deprecated and will be removed in a future release. Please use GLiNER.inference instead.
  pred_batches = model.batch_predict_entities(texts, LABELS, threshold=threshold)


[{'entities': [{'confidence': 0.8229873776435852,
                'end': 25,
                'label': 'ACTOR',
                'source': 'gliner',
                'start': 0,
                'text': 'Người điều khiển xe mô tô'},
               {'confidence': 0.7422376275062561, 'end': 47, 'label': 'ACTOR', 'source': 'gliner', 'start': 32, 'text': 'đội mũ bảo hiểm'}],
  'id': 0,
  'text': 'Người điều khiển xe mô tô không đội mũ bảo hiểm khi tham gia giao thông.'},
 {'entities': [{'confidence': 0.8878093957901001,
                'end': 16,
                'label': 'DOCUMENT',
                'source': 'gliner',
                'start': 0,
                'text': 'Giấy phép lái xe'}],
  'id': 1,
  'text': 'Giấy phép lái xe phải còn thời hạn sử dụng và phù hợp với loại xe được điều khiển.'},
 {'entities': [{'confidence': 0.9115290641784668, 'end': 7, 'label': 'VEHICLE', 'source': 'gliner', 'start': 0, 'text': 'Xe ô tô'}],
  'id': 2,
  'text': 'Xe ô tô không chấp hành hiệu lệnh của đèn tín